In [4]:
#Required imports
import pandas as pd
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
openpyxl.__version__


'3.1.5'

In [2]:

# Filepath
DATA_DIR = "../../data"
files = [
    "2016 ATL311 Open Records Request - Shreya Chivilkar",
    "2017 ATL311 Open Records Request - Shreya Chivilkar",
    "2018 ATL311 Open Records Request - Shreya Chivilkar",
    "2019 ATL311 Open Records Request - Shreya Chivilkar",
    "2020 ATL311 Open Records Request - Shreya Chivilkar",
    "2021 - A ATL311 Open Records Request - Shreya Chivilkar",
    "2021 - B ATL311 Open Records Request - Shreya Chivilkar",
    "2022 ATL311 Open Records Request - Shreya Chivilkar",
    "2023 ATL311 Open Records Request - Shreya Chivilkar",
    "2024 ATL311 Open Records Request - Shreya Chivilkar",
    "2025 ATL311 Open Records Request - Shreya Chivilkar",
]

for file in files:
    file_path = os.path.join(DATA_DIR, file + ".xlsx")

    print("\n" + "=" * 80)
    print(f"Reading file: {file}")

    try:
        df = pd.read_excel(
            file_path,
            engine="openpyxl"
        )

        print(f"Shape: {df.shape}")
        print("Columns:")
        for col in df.columns:
            print(f" - {col}")

    except Exception as e:
        print(f"❌ Error reading file: {e}")



Reading file: 2016 ATL311 Open Records Request - Shreya Chivilkar
Shape: (505970, 13)
Columns:
 - Opened
 - Description
 - Street #
 - Street Prefix
 - Street Name
 - Street Type
 - Street Suffix
 - Cross Street 2
 - Cross Street 2 Type
 - Postal Code
 - Closed Date
 - Status
 - Service Request #

Reading file: 2017 ATL311 Open Records Request - Shreya Chivilkar
Shape: (521542, 13)
Columns:
 - Opened
 - Description
 - Street #
 - Street Prefix
 - Street Name
 - Street Type
 - Street Suffix
 - Cross Street 2
 - Cross Street 2 Type
 - Postal Code
 - Closed Date
 - Status
 - SR #

Reading file: 2018 ATL311 Open Records Request - Shreya Chivilkar
Shape: (478077, 13)
Columns:
 - Opened
 - Attached KBA
 - Street #
 - Street Prefix
 - Street Name
 - Street Type
 - Street Suffix
 - Cross Street 2
 - Cross Street 2 Type
 - Postal Code
 - Call Center Closed Dt
 - Status
 - Service Request #

Reading file: 2019 ATL311 Open Records Request - Shreya Chivilkar
Shape: (619156, 13)
Columns:
 - Opened

In [ ]:
df = pd.read_excel(
    "../../data/2025 ATL311 Open Records Request - Shreya Chivilkar.xlsx",
    engine="openpyxl"
)

df.head()
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 379022 entries, 0 to 379021
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Opened             379022 non-null  datetime64[ns]
 1   Short Description  379022 non-null  object        
 2   Address            131749 non-null  object        
 3   Zip Code           119846 non-null  object        
 4   Closed Date 1      0 non-null       float64       
 5   Closed Date 2      357795 non-null  datetime64[ns]
 6   Status             379022 non-null  object        
 7   Number             379022 non-null  object        
dtypes: datetime64[ns](2), float64(1), object(5)
memory usage: 23.1+ MB


In [3]:
df['Closed Date 1'] = pd.to_datetime(df['Closed Date 1'], errors='coerce')
df['Closed Date 2'] = pd.to_datetime(df['Closed Date 2'], errors='coerce')
print(df.head())

               Opened                                  Short Description  \
0 2025-01-01 07:24:00                   Damaged Garbage Cart Replacement   
1 2025-01-01 09:44:00                   Entire Street Missed - Recycling   
2 2025-01-01 10:26:00                                    Illegal Dumping   
3 2025-01-01 11:08:00  Scooter and Bike Removal Requests (Shareable D...   
4 2025-01-01 11:50:00                                       Cart Pick Up   

                                       Address Zip Code Closed Date 1  \
0        3430 HOGAN RD SW , ATLANTA, GA, 30331    30331           NaT   
1                 Adair Ave NE, Atlanta, 30306    30306           NaT   
2    171 CHICAMAUGA PL SW , ATLANTA, GA, 30314    30314           NaT   
3  Baker Street, Luckie Street, Atlanta, 30313    30313           NaT   
4     2118 MEMORIAL DR SE , ATLANTA, GA, 30317    30317           NaT   

        Closed Date 2    Status      Number  
0 2025-01-09 13:24:00  Resolved  CS10032906  
1 2025-01-02

**Some records have multiple closure timestamps. Data providers suggested to select the closure date closest to the request’s opened time. If only one closure date exists, we use that; if both exist, we choose the earliest valid resolution relative to opening. This ensures consistent resolution-time estimates.**

In [ ]:
def select_closure_date(row):
    opened = row['Opened']
    c1 = row['Closed Date 1']
    c2 = row['Closed Date 2']
    
    # If both are missing
    if pd.isna(c1) and pd.isna(c2):
        return pd.NaT
    
    # If only one exists
    if pd.isna(c1):
        return c2
    if pd.isna(c2):
        return c1
    
    # If both exist, choose the one closer to Opened
    if abs((c1 - opened).total_seconds()) <= abs((c2 - opened).total_seconds()):
        return c1
    else:
        return c2
df['closed'] = df.apply(select_closure_date, axis=1)


In [ ]:
df['resolution_time_hours'] = (
    df['closed'] - df['Opened']
).dt.total_seconds() / 3600

df[['Closed Date 1', 'Closed Date 2', 'closed']].notna().sum()

resolved_rate = df['resolution_time_hours'].notna().mean()
print(f"{resolved_rate:.2%} of requests are resolved")


94.40% of requests are resolved


In [8]:
status_resolved_rate = (df['Status'] == 'Resolved').mean()
timestamp_resolved_rate = df['resolution_time_hours'].notna().mean()
print("Status based resoltuion rate:", status_resolved_rate)
print("Timestamp based resolution rate:", timestamp_resolved_rate)


Status based resoltuion rate: 0.937876957010411
Timestamp based resolution rate: 0.9439953353631187


In [9]:
df['has_closure_time'] = df['resolution_time_hours'].notna()

pd.crosstab(
    df['Status'],
    df['has_closure_time'],
    margins=True
)


has_closure_time,False,True,All
Status,,,
Active,21216,0,21216
Cancelled,0,2330,2330
Resolved,11,355465,355476
All,21227,357795,379022


Active cases behave exactly as expected : 100% of Active cases have no closure time.

Cancelled != unresolved, they have a time-to-closure but we shouldn’t treat it as successful service.

11 out of 355,476 resolved cases are missing closure timestamps
Resolved with timestamps: 355,465 / 379,022 ≈ 93.8%

Data inconsistency rate:
11 / 379,022 ≈ 0.003%

The dataset has high internal consistency for year 2025



In [10]:
resolved_df = df[
    (df['Status'] == 'Resolved') &
    (df['resolution_time_hours'].notna())
]


For Resolution-Time Metrics exclude: Active, Cancelled , The 11 inconsistent rows

Cancelled cases reflect administrative closure, not service delivery, so we exclude them from responsiveness metrics.

In [11]:
resolved_df = df[
    (df['Status'] == 'Resolved') &
    (df['resolution_time_hours'].notna()) &
    (df['resolution_time_hours'] > 0)
].copy()


**Filtered data to focus only on Atlanta/ Georgia**

In [ ]:
resolved_df = resolved_df.copy()
resolved_df['Zip Code'] = resolved_df['Zip Code'].astype(str)

resolved_df['zip_clean'] = (
    resolved_df['Zip Code']
    .str.extract(r'(\b\d{5}\b)', expand=False)
)
resolved_df['zip_int'] = pd.to_numeric(
    resolved_df['zip_clean'],
    errors='coerce'
)
resolved_df_ga = resolved_df[
    (
        (resolved_df['zip_int'] >= 30000) & (resolved_df['zip_int'] <= 31999)
    ) |
    (
        (resolved_df['zip_int'] >= 39800) & (resolved_df['zip_int'] <= 39999)
    )
].copy()

len(resolved_df), len(resolved_df_ga)


(351942, 100254)

In [17]:
resolved_df_ga.head()

,Opened,Short Description,Address,Zip Code,Closed Date 1,Closed Date 2,Status,Number,closed,resolution_time_hours,has_closure_time,zip_clean,zip_int
0,2025-01-01 07:24:00,Damaged Garbage Cart Replacement,"3430 HOGAN RD SW , ATLANTA, GA, 30331",30331,NaT,2025-01-09 13:24:00,Resolved,CS10032906,2025-01-09 13:24:00,198.000000,True,30331,30331.0
1,2025-01-01 09:44:00,Entire Street Missed - Recycling,"Adair Ave NE, Atlanta, 30306",30306,NaT,2025-01-02 08:30:00,Resolved,CS10032907,2025-01-02 08:30:00,22.766667,True,30306,30306.0
2,2025-01-01 10:26:00,Illegal Dumping,"171 CHICAMAUGA PL SW , ATLANTA, GA, 30314",30314,NaT,2025-01-17 08:10:00,Resolved,CS10032909,2025-01-17 08:10:00,381.733333,True,30314,30314.0
3,2025-01-01 11:08:00,Scooter and Bike Removal Requests (Shareable D...,"Baker Street, Luckie Street, Atlanta, 30313",30313,NaT,2025-01-02 07:42:00,Resolved,CS10032910,2025-01-02 07:42:00,20.566667,True,30313,30313.0
4,2025-01-01 11:50:00,Cart Pick Up,"2118 MEMORIAL DR SE , ATLANTA, GA, 30317",30317,NaT,2025-01-09 15:12:00,Resolved,CS10032912,2025-01-09 15:12:00,195.366667,True,30317,30317.0


In [ ]:
resolved_df_ga.to_csv(
    "../../data/cleaned_data/cleaned_data_2025.csv",
    index=False
)

#### **Script to cleanup and preprocess all the files that need to be analyzed!**


In [5]:
##Required methods:
def extract_year(filename):
    match = re.search(r'20\d{2}', filename)
    return match.group(0) if match else "unknown"

def select_closure_date(row):
    opened = row['Opened']
    c1 = row['Closed Date 1']
    c2 = row['Closed Date 2']
    
    # If both are missing
    if pd.isna(c1) and pd.isna(c2):
        return pd.NaT
    
    # If only one exists
    if pd.isna(c1):
        return c2
    if pd.isna(c2):
        return c1
    
    # If both exist, choose the one closer to Opened
    if abs((c1 - opened).total_seconds()) <= abs((c2 - opened).total_seconds()):
        return c1
    else:
        return c2
    
#Pipeline

DATA_DIR = "../../data"
OUTPUT_DIR = "../../data/cleaned_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

files = [
    "2021 - B ATL311 Open Records Request - Shreya Chivilkar.xlsx",
    "2022 ATL311 Open Records Request - Shreya Chivilkar.xlsx",
    "2023 ATL311 Open Records Request - Shreya Chivilkar.xlsx",
    "2024 ATL311 Open Records Request - Shreya Chivilkar.xlsx",
    "2025 ATL311 Open Records Request - Shreya Chivilkar.xlsx",
]

for file in files:
    print("\n" + "=" * 80)
    print(f"Processing {file}")

    year = extract_year(file)
    path = os.path.join(DATA_DIR, file)

    df = pd.read_excel(path, engine="openpyxl")

    # --- Datetime parsing ---
    df['Opened'] = pd.to_datetime(df['Opened'], errors='coerce')
    df['Closed Date 1'] = pd.to_datetime(df['Closed Date 1'], errors='coerce')
    df['Closed Date 2'] = pd.to_datetime(df['Closed Date 2'], errors='coerce')

    # --- Closure logic ---
    df['closed'] = df.apply(select_closure_date, axis=1)

    # --- Resolution time ---
    df['resolution_time_hours'] = (
        df['closed'] - df['Opened']
    ).dt.total_seconds() / 3600

    # --- Resolution metrics ---
    resolved_rate = df['resolution_time_hours'].notna().mean()
    status_resolved_rate = (df['Status'] == 'Resolved').mean()

    print(f"Timestamp resolved rate: {resolved_rate:.2%}")
    print(f"Status resolved rate: {status_resolved_rate:.2%}")

    # --- Closure availability ---
    df['has_closure_time'] = df['resolution_time_hours'].notna()

    # --- Filter resolved valid records ---
    resolved_df = df[
        (df['Status'] == 'Resolved') &
        (df['resolution_time_hours'].notna()) &
        (df['resolution_time_hours'] > 0)
    ].copy()

    # --- ZIP cleaning ---
    resolved_df['Zip Code'] = resolved_df['Zip Code'].astype(str)

    resolved_df['zip_clean'] = (
        resolved_df['Zip Code']
        .str.extract(r'(\b\d{5}\b)', expand=False)
    )

    resolved_df['zip_int'] = pd.to_numeric(
        resolved_df['zip_clean'],
        errors='coerce'
    )

    # --- GA ZIP filtering ---
    resolved_df_ga = resolved_df[
        (
            (resolved_df['zip_int'] >= 30000) & (resolved_df['zip_int'] <= 31999)
        ) |
        (
            (resolved_df['zip_int'] >= 39800) & (resolved_df['zip_int'] <= 39999)
        )
    ].copy()

    print("Resolved rows:", len(resolved_df))
    print("GA filtered rows:", len(resolved_df_ga))

    # --- Save cleaned file ---
    output_path = os.path.join(
        OUTPUT_DIR,
        f"cleaned_ATL311_{year}.csv"
    )

    resolved_df_ga.to_csv(output_path, index=False)

    print(f"Saved → {output_path}")



Processing 2021 - B ATL311 Open Records Request - Shreya Chivilkar.xlsx
Timestamp resolved rate: 96.95%
Status resolved rate: 97.25%
Resolved rows: 294179
GA filtered rows: 104791
Saved → ../../data/cleaned_data\cleaned_ATL311_2021.csv

Processing 2022 ATL311 Open Records Request - Shreya Chivilkar.xlsx
Timestamp resolved rate: 93.24%
Status resolved rate: 99.12%
Resolved rows: 221647
GA filtered rows: 52314
Saved → ../../data/cleaned_data\cleaned_ATL311_2022.csv

Processing 2023 ATL311 Open Records Request - Shreya Chivilkar.xlsx
Timestamp resolved rate: 98.61%
Status resolved rate: 98.55%
Resolved rows: 440536
GA filtered rows: 100594
Saved → ../../data/cleaned_data\cleaned_ATL311_2023.csv

Processing 2024 ATL311 Open Records Request - Shreya Chivilkar.xlsx
Timestamp resolved rate: 95.34%
Status resolved rate: 97.99%
Resolved rows: 358208
GA filtered rows: 98716
Saved → ../../data/cleaned_data\cleaned_ATL311_2024.csv

Processing 2025 ATL311 Open Records Request - Shreya Chivilkar.xl